
# Treatment Effect Analysis

This notebook evaluates the impact of email campaigns on customer revenue and purchase conversion using the Hillstrom randomized experiment.

- **Primary metric:** Revenue per randomized customer (`spend`)
- **Secondary metric:** Purchase conversion rate (`conversion`)
- **Diagnostic metric:** Visit rate (`visit`)
- **Control group:** No E-Mail

Detailed KPI calculations and treatment effect estimates are available in `sql/04_experiment_outcomes.sql`.



## 1. Observed Treatment Effects

The observed purchase conversion rates were 0.57% for Control, 1.25% for Men's Email, and 0.88% for Women's Email.

| Metric | Men's Email vs Control | Women's Email vs Control |
|---|---:|---:|
| Incremental Revenue per Customer | +$0.7698 | +$0.4244 |
| Relative Revenue Lift | +117.93% | +65.02% |
| Conversion Rate Lift | +0.68 pp | +0.31 pp |
| Visit Rate Lift | +7.66 pp | +4.52 pp |

Both email campaigns showed positive observed treatment effects, with Men's Email producing larger point estimates.

Although a 0.68 percentage-point increase may appear small, it represents an approximately 119% relative improvement over the control conversion rate of 0.57%.

Statistical inference is required to assess the uncertainty of these estimates.


## 2. Revenue Effect Analysis

Revenue per randomized customer (`spend`) is the primary metric.

Two-sided Welch's t-tests are used to compare mean spending between each email treatment and Control, allowing for unequal group variances.

**Hypotheses**

- H0: Mean spending is equal between treatment and control.
- H1: Mean spending differs between treatment and control.

**Significance level:** 0.05

Individual-level spending data are used to avoid intermediate rounding. Bootstrap confidence intervals will be calculated as an additional robustness check.

In [1]:
import pandas as pd
from scipy.stats import ttest_ind

# Load individual-level data
df = pd.read_csv("../data/raw/hillstrom.csv")

control_spend = df.loc[df["segment"] == "No E-Mail", "spend"]
men_spend = df.loc[df["segment"] == "Mens E-Mail", "spend"]
women_spend = df.loc[df["segment"] == "Womens E-Mail", "spend"]

# Welch's t-tests
result_men = ttest_ind(men_spend, control_spend, equal_var=False)
result_women = ttest_ind(women_spend, control_spend, equal_var=False)

# 95% confidence intervals
ci_men = result_men.confidence_interval(confidence_level=0.95)
ci_women = result_women.confidence_interval(confidence_level=0.95)

# Men's Email vs Control
print("Men's Email vs Control")
print(f"Incremental revenue: ${men_spend.mean() - control_spend.mean():.4f}")
print(f"T-statistic: {result_men.statistic:.4f}")
print(f"P-value: {result_men.pvalue:.6g}")
print(f"95% CI: (${ci_men.low:.4f}, ${ci_men.high:.4f})")

print()

# Women's Email vs Control
print("Women's Email vs Control")
print(f"Incremental revenue: ${women_spend.mean() - control_spend.mean():.4f}")
print(f"T-statistic: {result_women.statistic:.4f}")
print(f"P-value: {result_women.pvalue:.6g}")
print(f"95% CI: (${ci_women.low:.4f}, ${ci_women.high:.4f})")

Men's Email vs Control
Incremental revenue: $0.7698
T-statistic: 5.3001
P-value: 1.16381e-07
95% CI: ($0.4851, $1.0545)

Women's Email vs Control
Incremental revenue: $0.4244
T-statistic: 3.2564
P-value: 0.0011294
95% CI: ($0.1690, $0.6799)


### Revenue Hypothesis Test Results

| Comparison | Incremental Revenue | 95% CI | T-statistic | P-value |
|---|---:|---:|---:|---:|
| Men's Email vs Control | +$0.7698 | [$0.4851, $1.0545] | 5.3001 | 1.16381e-07 |
| Women's Email vs Control | +$0.4244 | [$0.1690, $0.6799] | 3.2564 | 0.0011294 |

**Interpretation:** Both email campaigns showed statistically significant increases in mean revenue per customer relative to Control (p < 0.05).

Both 95% confidence intervals exclude zero, supporting positive revenue effects. Men's Email produced a larger point estimate, but statistical superiority over Women's Email has not yet been established.

Given the substantial variability in individual spending, bootstrap confidence intervals will be used to further assess the robustness of these findings.

## 3. Conversion Rate Analysis

Two-sided two-proportion Z-tests are used to compare purchase conversion rates between each email treatment and Control.

**Hypotheses**

- H0: Treatment and control conversion rates are equal.
- H1: Treatment and control conversion rates differ.

**Significance level:** 0.05

In [2]:
from statsmodels.stats.proportion import proportions_ztest

# Men's Email vs Control
z_men, p_men = proportions_ztest(
    count=[267, 122],
    nobs=[21307, 21306],
    alternative="two-sided"
)

# Women's Email vs Control
z_women, p_women = proportions_ztest(
    count=[189, 122],
    nobs=[21387, 21306],
    alternative="two-sided"
)

print("Men's Email vs Control")
print(f"Z-statistic: {z_men:.4f}")
print(f"P-value: {p_men:.6g}")

print()

print("Women's Email vs Control")
print(f"Z-statistic: {z_women:.4f}")
print(f"P-value: {p_women:.6g}")

Men's Email vs Control
Z-statistic: 7.3851
P-value: 1.52322e-13

Women's Email vs Control
Z-statistic: 3.7796
P-value: 0.000157105


### Conversion Rate Hypothesis Test Results

| Comparison | Z-statistic | P-value |
|---|---:|---:|
| Men's Email vs Control | 7.3851 | 1.52322e-13 |
| Women's Email vs Control | 3.7796 | 0.000157105 |

**Interpretation:** Both email campaigns showed statistically significant increases in purchase conversion relative to Control (p < 0.05).

However, these results do not establish campaign profitability or demonstrate that Men's Email is statistically superior to Women's Email.

Confidence intervals for conversion rate differences and multiple-testing adjustments will be addressed in subsequent analyses.

## 4. Bootstrap Confidence Intervals for Revenue

Because customer spending is highly variable and concentrated among a small proportion of customers, bootstrap resampling is used as a robustness check for the primary metric.

Customers are resampled with replacement independently within each treatment and control group. The difference in mean revenue per customer is calculated for each bootstrap sample.

- Bootstrap iterations: 2,000
- Confidence level: 95%
- Method: Percentile bootstrap
- Random seed: 42

Bootstrap confidence intervals will be compared with the Welch confidence intervals.

In [5]:
import numpy as np

rng = np.random.default_rng(42)
n_bootstrap = 2000

# Men's Email vs Control
men_values = men_spend.to_numpy()
control_values = control_spend.to_numpy()

bootstrap_diffs_men = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    boot_men = rng.choice(
        men_values, size=len(men_values), replace=True

    )
    boot_control = rng.choice(
        control_values, size=len(control_values), replace=True
    )

    bootstrap_diffs_men[i] = boot_men.mean() - boot_control.mean()

ci_boot_men = np.percentile(bootstrap_diffs_men, [2.5, 97.5])

# Women's Email vs Control
women_values = women_spend.to_numpy()

bootstrap_diffs_women = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    boot_women = rng.choice(
        women_values, size=len(women_values), replace=True
    )
    boot_control = rng.choice(
        control_values, size=len(control_values), replace=True
    )

    bootstrap_diffs_women[i] = boot_women.mean() - boot_control.mean()

ci_boot_women = np.percentile(bootstrap_diffs_women, [2.5, 97.5])

print(f"Observed effect: ${men_values.mean() - control_values.mean():.4f}")
print(f"Bootstrap 95% CI: (${ci_boot_men[0]:.4f}, ${ci_boot_men[1]:.4f})")
print(f"Welch 95% CI: (${ci_men.low:.4f}, ${ci_men.high:.4f})")

print()

print(f"Observed effect: ${women_values.mean() - control_values.mean():.4f}")
print(f"Bootstrap 95% CI: (${ci_boot_women[0]:.4f}, ${ci_boot_women[1]:.4f})")
print(f"Welch 95% CI: (${ci_women.low:.4f}, ${ci_women.high:.4f})")

Observed effect: $0.7698
Bootstrap 95% CI: ($0.4840, $1.0517)
Welch 95% CI: ($0.4851, $1.0545)

Observed effect: $0.4244
Bootstrap 95% CI: ($0.1712, $0.6854)
Welch 95% CI: ($0.1690, $0.6799)



### Bootstrap Results

| Comparison | Welch 95% CI | Bootstrap 95% CI |
|---|---|---|
| Men's Email vs Control | [$0.4851, $1.0545] | [$0.4840, $1.0517] |
| Women's Email vs Control | [$0.1690, $0.6799] | [$0.1712, $0.6854] |

**Interpretation:** The percentile bootstrap confidence intervals closely align with the Welch confidence intervals for both email campaigns.

All intervals exclude zero, supporting the conclusion that both campaigns increased mean revenue per randomized customer.

The agreement between methods provides an additional robustness check, although it does not eliminate potential sensitivity to extreme spending values.


## 5. Conversion Rate Confidence Intervals

Newcombe's method, based on Wilson score intervals, is used to estimate 95% confidence intervals for the difference in purchase conversion rates between each email treatment and Control.

In [6]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Men's Email vs Control
ci_conv_men = confint_proportions_2indep(
    count1=267,
    nobs1=21307,
    count2=122,
    nobs2=21306,
    compare="diff",
    method="newcomb",
    alpha=0.05
)

# Women's Email vs Control
ci_conv_women = confint_proportions_2indep(
    count1=189,
    nobs1=21387,
    count2=122,
    nobs2=21306,
    compare="diff",
    method="newcomb",
    alpha=0.05
)

print(
    f"Men's Email 95% CI: "
    f"({ci_conv_men[0] * 100:.4f}, {ci_conv_men[1] * 100:.4f}) pp"
)

print(
    f"Women's Email 95% CI: "
    f"({ci_conv_women[0] * 100:.4f}, {ci_conv_women[1] * 100:.4f}) pp"
)


Men's Email 95% CI: (0.5014, 0.8641) pp
Women's Email 95% CI: (0.1502, 0.4745) pp


### Conversion Rate Confidence Interval Results

| Comparison | Conversion Lift | Newcombe 95% CI |
|---|---:|---|
| Men's Email vs Control | +0.68 pp | [0.5014, 0.8641] pp |
| Women's Email vs Control | +0.31 pp | [0.1502, 0.4745] pp |

**Interpretation:** Both confidence intervals exclude zero, providing further evidence of positive conversion effects for both email campaigns.

These are individual 95% confidence intervals and have not been adjusted for multiple comparisons.

## 6. Multiple-Testing Adjustment

The Holm method is applied to the four revenue and conversion hypothesis tests to control the family-wise error rate at 0.05.

In [8]:
from statsmodels.stats.multitest import multipletests

p_values = [
    result_men.pvalue,
    result_women.pvalue,
    p_men,
    p_women
]

reject, p_adjusted, _, _ = multipletests(
    p_values,
    alpha=0.05,
    method="holm"
)

comparisons = [
    "Men's Email - Revenue",
    "Women's Email - Revenue",
    "Men's Email - Conversion",
    "Women's Email - Conversion"
]

for name, p_adj, significant in zip(comparisons, p_adjusted, reject):
    print(f"{name}")
    print(f"Adjusted p-value: {p_adj:.6g}")
    print(f"Significant: {significant}")
    print()

Men's Email - Revenue
Adjusted p-value: 3.49144e-07
Significant: True

Women's Email - Revenue
Adjusted p-value: 0.0011294
Significant: True

Men's Email - Conversion
Adjusted p-value: 6.09289e-13
Significant: True

Women's Email - Conversion
Adjusted p-value: 0.00031421
Significant: True



### Holm-Adjusted Results

| Comparison | Adjusted P-value | Significant? |
|---|---:|---|
| Men's Email – Revenue | 3.49144e-07 | Yes |
| Women's Email – Revenue | 0.0011294 | Yes |
| Men's Email – Conversion | 6.09289e-13 | Yes |
| Women's Email – Conversion | 0.00031421 | Yes |

**Interpretation:** All four comparisons remain statistically significant after Holm adjustment (α = 0.05).

These findings provide evidence of positive revenue and conversion effects for both email campaigns relative to Control, even after controlling the family-wise error rate across the four tests.

The four tests were treated as a single comparison family for this analysis. This adjustment does not establish statistical superiority of one email campaign over the other.

**Notes:**
- The previously reported 95% confidence intervals are individual, unadjusted intervals.
- The four-test family and Holm adjustment were specified during this secondary analysis, after inspecting initial results, rather than preregistered. The adjusted findings should therefore be interpreted as exploratory.

## 7. Direct Comparison Between Email Campaigns

Although Men's Email showed a larger estimated revenue lift than Women's Email, this does not necessarily imply a statistically significant difference between the two campaigns.

A two-sided Welch's t-test is used to directly compare mean revenue per randomized customer between the two email groups.

**Hypotheses**

- H0: Mean spending is equal between Men's Email and Women's Email.
- H1: Mean spending differs between Men's Email and Women's Email.

**Significance level:** 0.05

**Note:** This post hoc comparison was motivated by the observed campaign results and was not included in the earlier four-test Holm adjustment.

In [12]:
# Men's Email vs Women's Email
result_direct = ttest_ind(
    men_spend,
    women_spend,
    equal_var=False
)

ci_direct = result_direct.confidence_interval(
    confidence_level=0.95
)

print(f"Mean difference: ${men_spend.mean() - women_spend.mean():.4f}")
print(f"T-statistic: {result_direct.statistic:.4f}")
print(f"P-value: {result_direct.pvalue:.6g}")
print(f"95% CI: (${ci_direct.low:.4f}, ${ci_direct.high:.4f})")

Mean difference: $0.3454
T-statistic: 2.1640
P-value: 0.0304687
95% CI: ($0.0326, $0.6583)


### Direct Comparison Results

| Comparison | Mean Difference | 95% CI | T-statistic | P-value |
|---|---:|---|---:|---:|
| Men's vs Women's Email | +$0.3454 | [$0.0326, $0.6583] | 2.1640 | 0.0304687 |

**Interpretation:** Men's Email generated statistically significantly higher mean revenue per customer than Women's Email in this exploratory, unadjusted comparison (p < 0.05).

The confidence interval excludes zero, although its lower bound is close to zero, indicating that the incremental advantage could be relatively small.